# 🧠 Modelo A — Classificador de Alta Recuperação

**Problema de negócio:** dado o histórico de pagamento de um contrato em cobrança, ele vai recuperar **acima da mediana da carteira**? Esta classificação ajuda a priorizar contratos no funil operacional.

Target: `total_pago >= mediana(total_pago)`. Binário (1 = alta recuperação, 0 = baixa).

Vamos descobrir o sinal dos dados antes de treinar.

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np
import joblib

GOLD = PROJECT / 'notebooks' / 'data' / 'gold'
MODELS = PROJECT / 'models'
MODELS.mkdir(exist_ok=True)

features = pd.read_parquet(GOLD / 'contract_features.parquet')
features.shape

(10000, 19)

## 1. Construir o target

Vamos olhar a distribuição de `total_pago` primeiro:

In [2]:
features['total_pago'].describe()

count    10000.000000
mean      5544.030928
std       2261.965191
min          0.000000
25%       3912.600000
50%       5340.615000
75%       6948.180000
max      15629.020000
Name: total_pago, dtype: float64

In [3]:
mediana = features['total_pago'].median()
features['alta_recuperacao'] = (features['total_pago'] >= mediana).astype(int)

print(f'Mediana: R$ {mediana:,.2f}')
print('Distribuição do target:')
print(features['alta_recuperacao'].value_counts(normalize=True).round(3))

Mediana: R$ 5,340.61
Distribuição do target:
alta_recuperacao
0    0.5
1    0.5
Name: proportion, dtype: float64


Balanceado em 50/50 (por construção da mediana). Bom para classificação.

## 2. Onde está o sinal?

Antes de jogar tudo no modelo, vamos ver quais features correlacionam com o target.

In [4]:
num_cols = features.select_dtypes('number').columns.drop(['alta_recuperacao', 'total_pago'])
corr = features[list(num_cols) + ['alta_recuperacao']].corr()['alta_recuperacao'].drop('alta_recuperacao')
corr.sort_values(ascending=False)

velocidade_pagamento    0.728322
parcelas_total          0.616409
parcelas_pagas          0.257128
media_dias_atraso       0.013467
valor_inadimplente      0.007629
score_risco            -0.021962
dias_atraso_inicial    -0.025478
taxa_adimplencia       -0.087276
Name: alta_recuperacao, dtype: float64

`velocidade_pagamento` e `parcelas_total` puxam o resultado para cima. `parcelas_pagas` também ajuda. Esses 3 vão carregar o sinal.

## 3. Features de entrada

Importante: **não usar `total_pago`** como feature — seria leakage direto. Mas `velocidade_pagamento` é derivada de `total_pago / dias_desde_primeiro_pagamento`, então também escapa. Vou usar features que descrevem *comportamento de pagamento observado*, que é o que o time operacional consegue ver no dia-a-dia.

In [5]:
NUMERIC = [
    'score_risco', 'dias_atraso_inicial', 'valor_inadimplente',
    'parcelas_total', 'parcelas_pagas',
    'taxa_adimplencia', 'media_dias_atraso', 'velocidade_pagamento',
]
CATEGORICAL = ['faixa_valor', 'dias_atraso_bucket', 'regiao',
               'nome_assessoria', 'metodo_predominante']
TARGET = 'alta_recuperacao'

data = features.dropna(subset=NUMERIC).copy()
X = data[NUMERIC + CATEGORICAL].copy()
y = data[TARGET]

for c in CATEGORICAL:
    X[c] = X[c].astype('category')

print('Shape:', X.shape, '| pos rate:', y.mean().round(3))

Shape: (8028, 13) | pos rate: 0.552


## 4. Treino/teste

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

((6422, 13), (1606, 13))

## 5. Treino XGBoost

In [7]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    enable_categorical=True,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print('Treino concluído.')

Treino concluído.


## 6. Avaliação

In [8]:
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, f1_score, accuracy_score)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f'Accuracy : {accuracy_score(y_test, pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, proba):.4f}')
print(f'F1-macro : {f1_score(y_test, pred, average="macro"):.4f}')
print()
print(classification_report(y_test, pred, target_names=['Baixa', 'Alta']))

Accuracy : 0.9041
ROC-AUC  : 0.9651
F1-macro : 0.9025

              precision    recall  f1-score   support

       Baixa       0.91      0.87      0.89       719
        Alta       0.90      0.93      0.92       887

    accuracy                           0.90      1606
   macro avg       0.91      0.90      0.90      1606
weighted avg       0.90      0.90      0.90      1606



In [9]:
cm = confusion_matrix(y_test, pred)
pd.DataFrame(cm,
             index=['real_baixa', 'real_alta'],
             columns=['pred_baixa', 'pred_alta'])

,pred_baixa,pred_alta
real_baixa,623,96
real_alta,58,829


## 7. Importância das features

In [10]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10)

velocidade_pagamento    0.475077
parcelas_total          0.190182
taxa_adimplencia        0.034253
nome_assessoria         0.032022
parcelas_pagas          0.031999
media_dias_atraso       0.031215
valor_inadimplente      0.030905
regiao                  0.030840
dias_atraso_bucket      0.030536
dias_atraso_inicial     0.029096
dtype: float32

## 8. Persistência

In [11]:
artifact = {
    'model': model,
    'numeric_features': NUMERIC,
    'categorical_features': CATEGORICAL,
    'target': 'alta_recuperacao',
    'target_threshold_brl': float(mediana),
    'version': '2.0.0',
    'metrics': {
        'accuracy': float(accuracy_score(y_test, pred)),
        'roc_auc':  float(roc_auc_score(y_test, proba)),
        'f1_macro': float(f1_score(y_test, pred, average='macro')),
    },
}
out = MODELS / 'collection_success_v1.joblib'
joblib.dump(artifact, out)
print('Salvo em', out)
print('Métricas:', artifact['metrics'])

Salvo em /home/ivissonalves/Workspaces/previsia/previsia-api/models/collection_success_v1.joblib
Métricas: {'accuracy': 0.9041095890410958, 'roc_auc': 0.9651244290501182, 'f1_macro': 0.9025055187637969}
